# ATHAR Chat: LLM baselines on Kaggle GPUs (S4: Jais 2 / Nile-Chat / Qwen3, S7: Jais 2 + QLoRA)

The LLM predicts **only the intent label** (`create_reminder`, `create_alarm`, `unsupported`), like every
other system. Routing, argument extraction and the replies come from the same shared, deterministic
`athar` pipeline, and the scores come from the same pre-registered harness
(`docs/evaluation_protocol.md`). So these numbers are directly comparable to S1–S6.

| System | What | Status in the protocol |
|---|---|---|
| S4 | instruction-tuned LLM, zero- and few-shot, three scoring methods (below) | extended, reported separately |
| S7 | Jais 2 8B + QLoRA fine-tuned on the train split | exploratory |

**Scoring methods (all reported; `calibrated` is the primary S4 number, fixed in advance):**

| Method | How it works |
|---|---|
| `raw` | constrained label scoring: summed log-probability of each Arabic verbalizer (`تذكير`, `منبه`, `غير مدعوم`) after the chat prompt, as in lm-evaluation-harness `multiple_choice` |
| `calibrated` | contextual calibration (Zhao et al. 2021): divide by the label probabilities the model gives to content-free inputs |
| `generate` | greedy generation, parsed; an unparseable answer counts as a failure |

### Setup (once)
1. **Build the kit locally:** run `uv run python scripts/build_kaggle_kit.py`, then upload `dist/athar-chat-kit.zip` as a Kaggle **Dataset** named `athar-chat-kit`.
2. **Accept the Jais 2 terms:** open https://huggingface.co/inception42/Jais-2-8B-Chat and accept them (the model is gated).
3. **Add your token:** in the Kaggle notebook, go to *Add-ons → Secrets* and add `HF_TOKEN` (a read token).
4. **Set the notebook options:** *Accelerator* = **GPU T4 x2**, *Internet* = **on**, and attach the `athar-chat-kit` dataset.
5. **Run all.** It takes about 1.5–2 h on T4 ×2 for the three models on dev, including the robustness perturbations and MASSIVE. The optional S7 adds about 30 min. The Kaggle quota is about 30 GPU-hours per week.

**Test-set discipline:** `SPLIT = "dev"`. The test split is opened only once, in the Phase 4 run, with `UNLOCK_TEST = True`.

In [ ]:
# ---- configuration --------------------------------------------------------------------------
import os

# Kaggle mounts datasets at /kaggle/input/<slug>/ or /kaggle/input/datasets/<user>/<slug>/,
# sometimes one folder deeper: locate the kit by its wheel instead of hard-coding the path.
_wheels, _seen = [], []
for _root, _dirs, _files in os.walk("/kaggle/input", followlinks=True):   # dataset dirs can be symlinks
    _seen.append(f"{_root}: {sorted(_dirs + _files)[:8]}")
    _wheels += [os.path.join(_root, f) for f in _files if f.startswith("athar_chat-") and f.endswith(".whl")]
_wheels.sort()
if not _wheels:
    raise FileNotFoundError("athar_chat-*.whl not found under /kaggle/input. If you just uploaded a new "
                            "dataset version, wait until Kaggle finishes processing it, then in the right "
                            "panel remove and re-add athar-chat-kit (or pick its latest version) and "
                            "restart the session. Seen:\n" + "\n".join(_seen[:30]))
WHEEL = _wheels[-1]
KIT = os.path.dirname(WHEEL)
print("kit:", KIT)
MANIFEST = f"{KIT}/data/draft/manifest.json"     # real data later: f"{KIT}/data/splits/manifest.json"
SPLIT = "dev"                                     # "test" only in the Phase 4 run
UNLOCK_TEST = False
GATE_TAU = {}             # test run only: {system name: tau picked on dev}; empty on dev
SHOTS_PER_LABEL = 2                               # few-shot: 2 x 3 labels, from train only
OUT = "/kaggle/working/s4_results"

# (key, Hugging Face id, precision on 2x T4, chat-template kwargs)
MODELS = [
    ("jais2-8b",  "inception42/Jais-2-8B-Chat", "fp16", {}),   # falls back to nf4-fp32 on fp16 overflow
    ("nile-4b",   "MBZUAI-Paris/Nile-Chat-4B",  "fp32", {}),   # Gemma-3 overflows in fp16 on T4
    ("qwen3-8b",  "Qwen/Qwen3-8B",              "fp16", {"enable_thinking": False}),
]
RUN_S4 = True             # set False to rerun only S7 / MASSIVE
RUN_S7_QLORA = True       # S7: fine-tune Jais 2 (QLoRA) on the train split, 178 of 260 items project management
RUN_MASSIVE_OOD = True    # out-of-distribution intent check on MASSIVE ar-SA
RUN_EGYPTIAN_OOD = True   # native-written Egyptian out-of-scope check (Alhasan/egyptian-nlu, CC BY 4.0)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q --no-deps --ignore-requires-python "{WHEEL}"
!pip install -q "transformers>=5.0" accelerate bitsandbytes peft "tzdata>=2024.1" "pydantic>=2.7"

In [ ]:
import gc
import json
import time
from pathlib import Path

import pandas as pd
import torch
import yaml
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

# Gated models (Jais 2) need the terms accepted *and* the request approved on the model page,
# for the account that owns HF_TOKEN. Skip any model the token cannot read instead of failing mid-run.
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError


def _can_read(model_id):
    try:
        hf_hub_download(model_id, "config.json", token=HF_TOKEN)
        return True
    except (GatedRepoError, RepositoryNotFoundError) as e:
        print(f"SKIPPING {model_id}: no access with this HF_TOKEN ({type(e).__name__}). "
              f"Request access at https://huggingface.co/{model_id} and rerun once approved.")
        return False

MODELS = [m for m in MODELS if _can_read(m[1])]
if RUN_S7_QLORA and not _can_read("inception42/Jais-2-8B-Chat"):
    RUN_S7_QLORA = False
print("models to run:", [m[0] for m in MODELS])

from athar import CONTRACT_VERSION
from athar.data.split import load_split, read_manifest
from athar.eval import harness as H
from athar.intent import llm
from athar.intent.rules import KeywordIntent

CFG = yaml.safe_load(open(f"{KIT}/configs/eval.yaml", encoding="utf-8"))
CFG["oracle"] = False
CFG["comparisons"] = []          # S4 is reported separately: no tests against the core systems
train = load_split(MANIFEST, "train")
items = load_split(MANIFEST, SPLIT, unlock_test=UNLOCK_TEST)
SHOTS = llm.select_shots(train, per_label=SHOTS_PER_LABEL, seed=CFG["seed"])
Path(OUT).mkdir(parents=True, exist_ok=True)
print(f"contract v{CONTRACT_VERSION}; manifest {read_manifest(MANIFEST)['manifest_sha256'][:12]}; "
      f"train {len(train)}, {SPLIT} {len(items)}; {len(SHOTS)} shots:", [lab for _, lab in SHOTS])

In [ ]:
FALLBACK = "nf4-fp32"   # 4-bit weights, fp32 compute: cannot overflow, fits an 8B model on 2x T4


def load_checked(model_id, precision):
    # Load and make sure a forward pass is finite; fall back to 4-bit/fp32-compute if fp16 overflows.
    model, tok = llm.load_llm(model_id, precision, token=HF_TOKEN)
    probe = llm.LLMIntent(model, tok, shots=SHOTS)
    try:
        probe.label_logprobs("فكرني بكره الساعه 7 الصبح اشتري عيش")
        return model, tok, precision
    except FloatingPointError as e:
        print("  ", e, f"-> reloading in {FALLBACK}")
        del model, probe; gc.collect(); torch.cuda.empty_cache()
        model, tok = llm.load_llm(model_id, FALLBACK, token=HF_TOKEN)
        return model, tok, FALLBACK


def build_recognizers(model, tok, key, tkw):
    recognizers = {}
    for shots_name, shots in (("0shot", []), (f"{len(SHOTS)}shot", SHOTS)):
        shared = None  # same prompt + shots -> one cache of label scores for all three methods
        for method in ("raw", "calibrated", "generate"):
            name = f"S4-{key}-{shots_name}-{method}"
            r = llm.LLMIntent(model, tok, name=name, shots=shots, method=method, template_kwargs=tkw)
            if shared is None:
                shared = r
            else:
                r._cache, r._cf = shared._cache, None
            recognizers[name] = r
    return recognizers


def summarise(name, rep):
    rep = {**rep, "name": name}
    m, ci = rep["metrics"], rep["ci"]["safe_task_success"]
    cal = rep.get("calibration") or {}
    pm = rep["subgroups"]["domain"]["project_management"]
    return {"system": name, "STS": round(m["safe_task_success"], 3), "STS_low": round(ci["low"], 3),
            "STS_high": round(ci["high"], 3), "intent_F1": round(m["intent_macro_f1"], 3),
            "unsafe": round(m["unsafe_execution_rate"], 3), "invalid": m["invalid_outputs"],
            "ECE": round(cal.get("ece", float("nan")), 3),
            # project managers are the main target users: their requests scored on their own
            "STS_PM": round(pm["safe_task_success"]["estimate"], 3),
            "F1_PM": round(pm["intent_macro_f1"]["estimate"], 3),
            **gate_cols(rep)}


def gate_cols(rep):
    # confidence gate (protocol 6): the tau the dev rule picks, and STS / unsafe execution there.
    # On dev this is the selection itself; for the test run, fix GATE_TAU from the dev run.
    g = rep.get("gate")
    if g is None:
        return {}
    tau = GATE_TAU.get(rep.get("name"), g["rule_threshold_on_this_split"])
    row = next(r for r in g["curve"] if r["tau"] == tau)
    return {"gate_tau": tau, "STS_gated": round(row["safe_task_success"], 3),
            "unsafe_gated": round(row["unsafe_execution_rate"], 3), "asks_confirm": round(row["gated"], 3)}


rows = []
for key, model_id, precision, tkw in (MODELS if RUN_S4 else []):
    t0 = time.time()
    model, tok, used = load_checked(model_id, precision)
    sha = HfApi().model_info(model_id, token=HF_TOKEN).sha
    try:
        recognizers = build_recognizers(model, tok, key, tkw)
        report = H.evaluate(recognizers, items, CFG)
    except FloatingPointError as e:
        # The probe passed but a later input overflowed: rerun the whole model in the fallback,
        # so every number for this model comes from one precision.
        if used == FALLBACK:
            raise
        print("  ", e, f"-> rerunning {key} in {FALLBACK}")
        recognizers = None; del model; gc.collect(); torch.cuda.empty_cache()
        model, tok = llm.load_llm(model_id, FALLBACK, token=HF_TOKEN)
        used = FALLBACK
        recognizers = build_recognizers(model, tok, key, tkw)
        report = H.evaluate(recognizers, items, CFG)
    report["model"] = {"id": model_id, "revision": sha, "precision": used, "split": SPLIT,
                       "minutes": round((time.time() - t0) / 60, 1)}
    Path(f"{OUT}/{key}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str))
    rows += [summarise(n, r) for n, r in report["systems"].items()]
    print(pd.DataFrame(rows).tail(len(recognizers)).to_string(index=False))
    del model, recognizers; gc.collect(); torch.cuda.empty_cache()

summary = pd.DataFrame(rows)
if RUN_S4:
    summary.to_csv(f"{OUT}/summary.csv", index=False)
summary

## S7 (exploratory): Jais 2 8B + QLoRA on the train split

This is the run that trains the model on project-management requests: the train split is 260
items, 178 of them (68%) from project managers (standups, sprints, client follow-ups, Jira,
go-lives…) and 82 everyday requests. Results include `STS_PM` / `F1_PM`, the scores on the
project-manager requests alone.

The model is fine-tuned on the zero-shot prompt → verbalizer task, with the loss on the label tokens
only. It is then scored like S4 (`raw` and `calibrated`, zero-shot prompt).

Fixed settings: 4-bit NF4 (fp32 compute, since Jais 2 overflows in fp16 on T4), LoRA r = 16, α = 32, dropout 0.05 on all linear layers, LR 2e-4, 3 epochs, seed 0.
There is no dev search.

In [ ]:
if RUN_S7_QLORA:
    t0 = time.time()
    base, tok = llm.load_llm("inception42/Jais-2-8B-Chat", "nf4-fp32", token=HF_TOKEN)
    tuned = llm.lora_finetune(base, tok, train, epochs=3, learning_rate=2e-4, r=16, alpha=32,
                              dropout=0.05, seed=0)
    s7 = {f"S7-jais2-qlora-{m}": llm.LLMIntent(tuned, tok, name=f"S7-jais2-qlora-{m}", method=m)
          for m in ("raw", "calibrated")}
    rep7 = H.evaluate(s7, items, CFG)
    rep7["model"] = {"id": "inception42/Jais-2-8B-Chat", "adapter": "QLoRA r16", "precision": "nf4-fp32", "split": SPLIT,
                     "minutes": round((time.time() - t0) / 60, 1)}
    Path(f"{OUT}/s7_jais2_qlora.json").write_text(json.dumps(rep7, ensure_ascii=False, indent=1, default=str))
    print(pd.DataFrame([summarise(n, r) for n, r in rep7["systems"].items()]).to_string(index=False))
    del base, tuned, s7; gc.collect(); torch.cuda.empty_cache()   # free the GPUs for the MASSIVE check

## Out-of-distribution check: MASSIVE ar-SA (intent only)

MASSIVE (Amazon, CC BY 4.0) is Saudi-localised MSA, not Egyptian, and it has no datetime gold. This
check therefore uses **intent labels only** and is reported separately.

**Label mapping:**

| MASSIVE intent | Our label |
|---|---|
| `alarm_set` | `create_alarm` |
| `alarm_remove`, `calendar_remove` | `unsupported` (deletion) |
| `weather_query`, `play_music`, `email_sendemail`, `calendar_query` | `unsupported` (other tool) |
| `calendar_set` | **excluded** |

`calendar_set` is excluded because it mixes one-off reminders with calendar events, which our contract
treats as another tool. A balanced sample of up to 100 per mapped label is used (seed 0).

S1 (rules) is included as a reference because it needs no GPU.

In [ ]:
if RUN_MASSIVE_OOD:
    from datasets import load_dataset
    from sklearn.metrics import f1_score

    from athar.normalize import norm

    ds = load_dataset("mteb/amazon_massive_intent", "ar", split="test")
    keep = {"alarm_set": "create_alarm", "alarm_remove": "unsupported", "calendar_remove": "unsupported",
            "weather_query": "unsupported", "play_music": "unsupported", "email_sendemail": "unsupported",
            "calendar_query": "unsupported"}
    label_col = "label_text" if "label_text" in ds.column_names else "label"
    df = pd.DataFrame({"text": ds["text"], "intent": ds[label_col]})
    df = df[df.intent.isin(keep)].assign(gold=lambda d: d.intent.map(keep))
    df = df.groupby("gold", group_keys=False).apply(lambda g: g.sample(min(len(g), 100), random_state=0))
    texts, gold = [norm(t) for t in df.text], list(df.gold)
    ood = [{"system": "S1", "macro_F1": f1_score(gold, KeywordIntent().predict(texts), average="macro")}]
    for key, model_id, precision, tkw in MODELS:
        model, tok, used = load_checked(model_id, precision)
        r = llm.LLMIntent(model, tok, name=key, shots=SHOTS, method="calibrated", template_kwargs=tkw)
        ood.append({"system": f"S4-{key}-{len(SHOTS)}shot-calibrated",
                    "macro_F1": f1_score(gold, r.predict(texts), average="macro")})
        del model, r; gc.collect(); torch.cuda.empty_cache()
    ood = pd.DataFrame(ood).round(3)
    ood.to_csv(f"{OUT}/massive_ood.csv", index=False)
    print(f"MASSIVE ar test, {len(df)} items:", df.gold.value_counts().to_dict())
    ood

## Out-of-distribution check: native-written Egyptian (Alhasan/egyptian-nlu)

The 200-item test split of `Alhasan/egyptian-nlu` (CC BY 4.0, derived from MASSIVE) was written and
annotated by hand by a native Egyptian speaker. The revision and SHA-256 are pinned in `athar.eval.ood`.
Same mapping as MASSIVE, giving 196 items. Only 4 of them set an alarm, so the main number is **unsupported recall** (n = 192):
does the model abstain on real out-of-scope Egyptian requests? For each wrong in-scope prediction, the
confidence is also reported (`caught@0.6`, `caught@0.7`: how many the confidence gate would have stopped).


In [ ]:
if RUN_EGYPTIAN_OOD:
    from athar.eval import ood as OOD
    from athar.gate import confidences
    from athar.normalize import norm

    rows_egy = OOD.load(f"{OUT}/egy_test.jsonl")          # downloads the pinned file once
    texts_egy = [norm(r["text"]) for r in rows_egy]
    egy = [{"system": "S1", **{k: v for k, v in OOD.evaluate(KeywordIntent(), rows_egy).items() if k != "errors"}}]
    for key, model_id, precision, tkw in MODELS:
        model, tok, used = load_checked(model_id, precision)
        r = llm.LLMIntent(model, tok, name=key, shots=SHOTS, method="calibrated", template_kwargs=tkw)
        res = OOD.evaluate(r, rows_egy)
        pred = r.predict(texts_egy)
        conf = confidences(r, texts_egy, pred)
        wrong = [c for c, p, x in zip(conf, pred, rows_egy, strict=True) if x["gold"] == "unsupported" and p != "unsupported"]
        egy.append({"system": f"S4-{key}-{len(SHOTS)}shot-calibrated", "precision": used,
                    **{k: v for k, v in res.items() if k != "errors"},
                    "caught@0.6": sum(c < 0.6 for c in wrong), "caught@0.7": sum(c < 0.7 for c in wrong)})
        Path(f"{OUT}/egyptian_ood_{key}_errors.json").write_text(
            json.dumps(res["errors"], ensure_ascii=False, indent=1))
        del model, r; gc.collect(); torch.cuda.empty_cache()
    egy = pd.DataFrame(egy).round(3)
    egy.to_csv(f"{OUT}/egyptian_ood.csv", index=False)
    print(OOD.ATTRIBUTION)
    egy


## Bring the results back

Download `/kaggle/working/s4_results/` (*Output* tab), then put it in the repo under `runs/s4/`
(dev) or `results/s4/` (the one Phase 4 test run).

Each JSON file has:
- every metric, with family-bootstrap CIs;
- robustness and calibration;
- the item-level scores;
- the model revision (`sha`) and the precision that was used.